# 05. Evaluating RAG Systems

This notebook turns evaluation into code and structured review.

We will score a small archive-style benchmark using both automatic heuristics and qualitative review fields.

The main lesson is that evaluation for archive assistants should not collapse everything into one generic score.


The workshop treats the system as retrieval-first:
- tokenization defines what the model sees
- retrieval quality often controls answer quality
- generators cannot reliably compensate for weak retrieval
- evaluation should include groundedness, citation, abstention, bilingual behavior, and community-review placeholders

Current notebook: `05_evaluating_rag_systems.ipynb`

This notebook treats evaluation as a multi-part process that includes retrieval precision, abstention, citation quality, hallucination checks, bilingual behavior, and review placeholders.

This is a core workshop notebook.

Workshop sequence:
1. `01_tokenization_playground.ipynb`
2. `02_embeddings_and_similarity.ipynb`
3. `03_retriever_benchmarking_for_rag.ipynb`
4. `04_llm_comparison_in_same_rag_pipeline.ipynb`
5. `05_evaluating_rag_systems.ipynb`
6. `06_optional_lora_or_domain_adaptation.ipynb` (optional)


## Quick concept refresher

- Tokenization turns text into units that models can process.
- Retrieval chooses which passages are available as evidence.
- The retriever selects context; the generator turns context into an answer.
- Evaluation in archive assistants is multi-layered because the system must be useful, grounded, reviewable, and appropriate.


In [ ]:
# Self-contained runtime setup for this notebook.
# Each notebook includes its own seed and device checks so it can run independently in Colab.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")
print(f"Detected device: {DEVICE}")
print(f"Seed set to: {SEED}")

NOTEBOOK_CONTEXT = {
    "seed": SEED,
    "device": DEVICE,
    "notebook": __name__ if "__name__" in globals() else "notebook",
    "framing": "retrieval-first archive assistant workshop",
}

NOTEBOOK_CONTEXT


In [ ]:
# In Colab, uncomment if needed.

# !pip -q install sentence-transformers transformers pandas scikit-learn

print("Evaluation notebook dependencies are ready once the required packages are installed.")


In [ ]:
# Imports.

import re
import textwrap

import pandas as pd
try:
    import torch
except ImportError:
    torch = None
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline


In [ ]:
# Reuse a small corpus with explicit ids so citations stay inspectable.

eval_corpus = [
    {"chunk_id": "E01", "text": "Restricted ceremonial recordings require community review before any access decision."},
    {"chunk_id": "E02", "text": "Public catalog summaries should cite the original source note when listing place-name spelling variants."},
    {"chunk_id": "E03", "text": "Winter-only stories should not be surfaced outside the approved seasonal context."},
    {"chunk_id": "E04", "text": "Transcript quality notes document OCR mistakes, missing diacritics, and merged speaker turns."},
    {"chunk_id": "E05", "text": "Governance metadata must be checked before showing kinship-based access materials to general users."},
]

eval_corpus_df = pd.DataFrame(eval_corpus)
eval_corpus_df


In [ ]:
# A hand-built evaluation set.
# We include expected support, abstention expectations, and a bilingual placeholder field.

evaluation_set = [
    {
        "example_id": "R01",
        "query": "Which records require community review before access?",
        "query_language": "en",
        "relevant_chunk_ids": ["E01"],
        "should_abstain": False,
        "expected_keywords": ["community review", "access"],
    },
    {
        "example_id": "R02",
        "query": "Quels résumés publics doivent citer la note source originale?",
        "query_language": "fr",
        "relevant_chunk_ids": ["E02"],
        "should_abstain": False,
        "expected_keywords": ["citer", "source"],
    },
    {
        "example_id": "R03",
        "query": "Can I show winter-only stories as general examples in summer?",
        "query_language": "en",
        "relevant_chunk_ids": ["E03"],
        "should_abstain": False,
        "expected_keywords": ["winter-only", "not"],
    },
    {
        "example_id": "R04",
        "query": "Which donor agreement permits public quotation of restricted ceremonies?",
        "query_language": "en",
        "relevant_chunk_ids": [],
        "should_abstain": True,
        "expected_keywords": [],
    },
    {
        "example_id": "R05",
        "query": "Find the note about OCR mistakes and missing diacritics.",
        "query_language": "en",
        "relevant_chunk_ids": ["E04"],
        "should_abstain": False,
        "expected_keywords": ["OCR", "diacritics"],
    },
]

eval_df = pd.DataFrame(evaluation_set)
eval_df


In [ ]:
# Build a small retriever and a small generator.
# The point here is to evaluate the whole retrieval-grounded loop, not to maximize performance.

retriever_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
eval_embeddings = retriever_model.encode(
    eval_corpus_df["text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    tokenizer="google/flan-t5-small",
    device=0 if (torch is not None and torch.cuda.is_available()) else -1,
)


In [ ]:
# Retrieval and generation helpers.
# These remain deliberately simple so participants can modify them live.

def retrieve_eval_context(query: str, top_k: int = 3):
    query_embedding = retriever_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, eval_embeddings)[0]
    ranked = eval_corpus_df.copy()
    ranked["score"] = scores
    return ranked.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


def build_eval_prompt(query: str, context_df: pd.DataFrame) -> str:
    evidence = "\n".join([f"[{row.chunk_id}] {row.text}" for row in context_df.itertuples()])
    return textwrap.dedent(
        f"""
        Answer using only the evidence below.
        If the evidence is insufficient, say ABSTAIN.
        Cite chunk ids in square brackets.

        Question: {query}

        Evidence:
        {evidence}

        Response format:
        Answer: ...
        Citations: [chunk ids]
        """
    ).strip()


def generate_eval_answer(query: str, context_df: pd.DataFrame):
    prompt = build_eval_prompt(query, context_df)
    return generator(prompt, max_new_tokens=96, do_sample=False)[0]["generated_text"]


In [ ]:
# Automatic metrics.
# These are intentionally lightweight heuristics.
# Real projects should combine them with human review and task-specific checks.

def parse_chunk_citations(text: str):
    return re.findall(r"\[(E\d+)\]", text)


def retrieval_precision(retrieved_ids, relevant_ids, k):
    return sum(doc_id in relevant_ids for doc_id in retrieved_ids[:k]) / max(k, 1)


def abstention_quality(answer: str, should_abstain: bool):
    abstained = "ABSTAIN" in answer.upper()
    return float(abstained == should_abstain)


def citation_correctness(answer: str, relevant_ids):
    if not relevant_ids:
        return float("ABSTAIN" in answer.upper())
    citations = parse_chunk_citations(answer)
    if not citations:
        return 0.0
    return float(set(citations).issubset(set(relevant_ids)))


def bilingual_quality_heuristic(answer: str, query_language: str):
    if query_language == "fr":
        french_markers = [" le ", " la ", " des ", " et ", " est ", " réponse", "source"]
        score = any(marker in f" {answer.lower()} " for marker in french_markers)
        return float(score)
    return 1.0


def hallucination_flag(answer: str, relevant_ids):
    citations = parse_chunk_citations(answer)
    if "ABSTAIN" in answer.upper():
        return 0
    if relevant_ids and not citations:
        return 1
    if citations and not set(citations).intersection(set(relevant_ids)):
        return 1
    return 0


In [ ]:
# Run the small benchmark.
# We log both scores and qualitative placeholders for later manual review.

rows = []

for example in evaluation_set:
    context_df = retrieve_eval_context(example["query"], top_k=3)
    answer = generate_eval_answer(example["query"], context_df)
    retrieved_ids = context_df["chunk_id"].tolist()

    rows.append(
        {
            "example_id": example["example_id"],
            "query": example["query"],
            "query_language": example["query_language"],
            "retrieved_ids": retrieved_ids,
            "answer": answer,
            "retrieval_precision_at_3": retrieval_precision(retrieved_ids, example["relevant_chunk_ids"], 3),
            "abstention_quality": abstention_quality(answer, example["should_abstain"]),
            "citation_correctness": citation_correctness(answer, example["relevant_chunk_ids"]),
            "bilingual_quality": bilingual_quality_heuristic(answer, example["query_language"]),
            "hallucination_flag": hallucination_flag(answer, example["relevant_chunk_ids"]),
            "community_review_status": "TO_REVIEW",
            "community_review_notes": "",
        }
    )

results_df = pd.DataFrame(rows)
results_df


In [ ]:
# Aggregate numeric metrics.
# Keep in mind that these are only the quantitative layer of evaluation.

numeric_columns = [
    "retrieval_precision_at_3",
    "abstention_quality",
    "citation_correctness",
    "bilingual_quality",
    "hallucination_flag",
]

summary = results_df[numeric_columns].mean().to_frame(name="mean_score")
summary


In [ ]:
# Qualitative review template.
# Archive assistants often need explicit human review fields, not just automatic metrics.

qualitative_review = results_df[
    [
        "example_id",
        "query",
        "answer",
        "retrieved_ids",
        "community_review_status",
        "community_review_notes",
    ]
].copy()

qualitative_review["citation_notes"] = ""
qualitative_review["support_notes"] = ""
qualitative_review["cultural_appropriateness_notes"] = ""
qualitative_review


## Exercises

- Mark which examples should abstain and inspect whether the model did so.
- Look for fluent but weakly supported answers.
- Design one more evaluation dimension tied to governance, permissions, or community review.
- Compare a citation failure with a retrieval failure. Which is easier to fix?
